# 04 — Modèle Transformer (from scratch)

**Prérequis** : `01_clustering.ipynb` exécuté.

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Sortie** : `../models/transformer_best.pt`, `../data/results_transformer.pkl`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

EMBED_DIM    = 128
NUM_HEADS    = 4     # EMBED_DIM doit être divisible par NUM_HEADS : 128/4=32
FF_DIM       = 256
NUM_BLOCKS   = 2
N_CLASSES    = 8
DROPOUT      = 0.1
MAX_LEN      = 64
BATCH_SIZE   = 64
LR           = 1e-3
N_EPOCHS     = 10
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement des données

In [ ]:
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} | Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')

## Bloc 3 — Dataset + DataLoader

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

loaders = {
    split: DataLoader(
        IntentDataset(data[split]),
        batch_size=BATCH_SIZE, shuffle=(split == 'train'), collate_fn=collate_fn
    )
    for split in ('train', 'val', 'test')
}
x, y = next(iter(loaders['train']))
print(f'Batch — textes: {x.shape}, labels: {y.shape}')

## Bloc 4 : Positional Encoding (pattern TP7)

In [ ]:
class PositionalEncoding(nn.Module):
    """Encode la position de chaque token dans la séquence via sin/cos."""
    def __init__(self, embed_dim, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe       = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, embed_dim)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

## Bloc 5 : TransformerBlock (pattern TP7)

In [ ]:
class TransformerBlock(nn.Module):
    """Un bloc Transformer : Multi-Head Attention + Feed-Forward + LayerNorm + Residual."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1     = nn.LayerNorm(embed_dim)
        self.norm2     = nn.LayerNorm(embed_dim)
        self.ff        = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        # Sub-layer 1 : Self-Attention avec connexion résiduelle (Pre-LN)
        x_norm   = self.norm1(x)
        attn_out, _ = self.attention(x_norm, x_norm, x_norm, key_padding_mask=padding_mask)
        x = x + self.dropout(attn_out)
        # Sub-layer 2 : Feed-Forward avec connexion résiduelle
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ff(x_norm))
        return x

## Bloc 6 : TransformerClassifier (pattern TP7)

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim,
                 num_blocks, output_dim, max_len=200, dropout=0.1):
        super(TransformerClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_enc   = PositionalEncoding(embed_dim, max_len)
        self.blocks    = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_blocks)
        ])
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(embed_dim, output_dim)

    def forward(self, x):
        padding_mask = (x == 0)  # True = position à ignorer
        # Embedding + Positional Encoding
        emb = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        emb = self.pos_enc(emb)
        emb = self.dropout(emb)
        # N blocs Transformer
        for block in self.blocks:
            emb = block(emb, padding_mask=padding_mask)
        # Mean pooling sur les tokens non-PAD
        mask_expanded = (~padding_mask).unsqueeze(-1).float()
        pooled = (emb * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        return self.fc(pooled)


model = TransformerClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_blocks=NUM_BLOCKS,
    output_dim=N_CLASSES,
    max_len=MAX_LEN,
    dropout=DROPOUT
).to(device)

print(model)
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')

## Bloc 7 : Entraînement (pattern TP7)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    for texts, labels in loaders['train']:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(texts), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_losses.append(total_loss / len(loaders['train']))

    model.eval()
    val_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for texts, labels in loaders['val']:
            texts, labels = texts.to(device), labels.to(device)
            out = model(texts)
            val_loss += criterion(out, labels).item()
            preds.extend(torch.argmax(out, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_losses.append(val_loss / len(loaders['val']))
    acc = accuracy_score(targets, preds)
    val_accs.append(acc)

    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | val_acc={acc:.4f}', end='')
    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(model.state_dict(), MODEL_DIR + 'transformer_best.pt')
        print(' ✓')
    else:
        print()

## Bloc 8 : Courbes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(val_accs, color='green')
axes[1].axhline(best_val_acc, color='red', linestyle='--', label=f'Best={best_val_acc:.4f}')
axes[1].set_title('Val Accuracy'); axes[1].legend()
plt.suptitle('Transformer — Courbes entraînement')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'transformer_curves.png', dpi=100)
plt.show()

## Bloc 9 : Évaluation sur le test set

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR + 'transformer_best.pt', map_location=device))
model.eval()
preds, targets = [], []
with torch.no_grad():
    for texts, labels in loaders['test']:
        texts, labels = texts.to(device), labels.to(device)
        preds.extend(torch.argmax(model(texts), dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())

test_acc = accuracy_score(targets, preds)
report   = classification_report(targets, preds, target_names=label_names)
print(f'Test Accuracy : {test_acc:.4f}\n{report}')

cm = confusion_matrix(targets, preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matrice de confusion — Transformer (acc={test_acc:.4f})')
plt.ylabel('Vrai'); plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'transformer_confusion.png', dpi=100)
plt.show()

## Bloc 10 : Sauvegarde des résultats

In [ ]:
results_transformer = {
    'model': 'Transformer', 'test_accuracy': test_acc, 'best_val_acc': best_val_acc,
    'train_losses': train_losses, 'val_losses': val_losses, 'val_accs': val_accs,
    'predictions': preds, 'targets': targets, 'report': report,
}
with open(DATA_DIR + 'results_transformer.pkl', 'wb') as f:
    pickle.dump(results_transformer, f)
print(f'Résultats → {DATA_DIR}results_transformer.pkl')
print(f'Modèle    → {MODEL_DIR}transformer_best.pt')
print(f'Test accuracy : {test_acc:.4f}')